In [1]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

class AgentType(Enum):
    PASSIVE = 1
    NORMAL = 2
    AGGRESSIVE = 3

TYPE_COEFFICIENTS = {
    AgentType.PASSIVE: 0.3,
    AgentType.NORMAL: 0.7,
    AgentType.AGGRESSIVE: 1.0
}

print("Imports successful!")
print(f"Mesa version: {mesa.__version__}")
print("V6 Scale Invariance: READY")

Imports successful!
Mesa version: 3.5.1
V6 Scale Invariance: READY


In [2]:
class QueueAgent(mesa.Agent):
    def __init__(self, model, agent_type):
        super().__init__(model)
        self.agent_type = agent_type
        self.type_coeff = TYPE_COEFFICIENTS[agent_type]
        self.urgency = np.random.uniform(0.3, 1.0)
        self.social_inhibition = np.random.uniform(0.2, 0.8)
        self.exited = False
        self.entry_time = None
        self.exit_time = None
        self.latency = None

    @property
    def behavior_score(self):
        base_score = (self.type_coeff * self.urgency /
                     self.social_inhibition)
        if self.pos:
            neighbors = self.model.grid.get_neighbors(
                self.pos,
                moore=True,
                include_center=False,
                radius=2
            )
            density_factor = 1.0 + (len(neighbors) * 0.05)
        else:
            density_factor = 1.0
        return base_score * density_factor

    def step(self):
        if self.exited:
            return
        if self.pos is None:
            return
        if self.entry_time is None:
            self.entry_time = self.model.steps

        x, y = self.pos
        move_prob = min(self.behavior_score, 1.0)

        if np.random.random() < move_prob:
            new_y = y - 1
            if new_y < 0:
                self.model.grid.remove_agent(self)
                self.exited = True
                self.exit_time = self.model.steps
                if self.entry_time is not None:
                    self.latency = (self.exit_time -
                                   self.entry_time)
                self.model.exited_count += 1
                self.model.exit_times.append(self.model.steps)
                if self.latency is not None:
                    self.model.all_latencies.append(
                        self.latency
                    )
                return

            new_pos = (x, new_y)
            cell_contents = self.model.grid.get_cell_list_contents(
                [new_pos]
            )
            max_occupancy = (
                3 if self.agent_type == AgentType.AGGRESSIVE
                else 2 if self.agent_type == AgentType.NORMAL
                else 1
            )
            if len(cell_contents) < max_occupancy:
                self.model.grid.move_agent(self, new_pos)


class BunchQueueModel(mesa.Model):
    def __init__(self, n_agents=100, width=20, height=30,
                 pct_aggressive=0.15, pct_normal=0.25):
        super().__init__()

        self.width = width
        self.height = height
        self.steps = 0
        self.exited_count = 0
        self.exit_times = []
        self.all_latencies = []
        self.total_agents = n_agents

        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )

        n_aggressive = int(n_agents * pct_aggressive)
        n_normal = int(n_agents * pct_normal)
        n_passive = n_agents - n_aggressive - n_normal

        agent_types = (
            [AgentType.AGGRESSIVE] * n_aggressive +
            [AgentType.NORMAL] * n_normal +
            [AgentType.PASSIVE] * n_passive
        )
        np.random.shuffle(agent_types)

        for agent_type in agent_types:
            agent = QueueAgent(self, agent_type)
            x = np.random.randint(0, width)
            y = np.random.randint(height // 2, height)
            self.grid.place_agent(agent, (x, y))

    def step(self):
        self.steps += 1
        self.agents.shuffle_do("step")

    def run(self, max_steps=500):
        for _ in range(max_steps):
            self.step()
            if self.exited_count >= self.total_agents:
                break
        return self.exited_count, self.exit_times


print("Agent and Model classes defined!")
print("Optimized for large scale runs")
print("No animation overhead")

Agent and Model classes defined!
Optimized for large scale runs
No animation overhead


In [3]:
# EXPERIMENT 1 - Aggression Sweep
# Test every aggression level 1% to 50%
# 30 runs per level to find precise optimal threshold

print("EXPERIMENT 1 - AGGRESSION SWEEP")
print("Testing aggression levels 1% to 50%")
print("30 runs per level...")
print("="*50)

aggression_levels = [i/100 for i in range(1, 51)]
n_runs = 30
sweep_results = []

for pct_agg in aggression_levels:
    run_exited = []
    run_steps = []
    run_latencies = []
    
    for run in range(n_runs):
        model = BunchQueueModel(
            n_agents=100,
            width=20,
            height=30,
            pct_aggressive=pct_agg,
            pct_normal=0.25
        )
        exited, _ = model.run(max_steps=500)
        run_exited.append(exited)
        run_steps.append(model.steps)
        if model.all_latencies:
            run_latencies.extend(model.all_latencies)
    
    avg_steps = np.mean(run_steps)
    std_steps = np.std(run_steps)
    avg_exited = np.mean(run_exited)
    avg_latency = np.mean(run_latencies) if run_latencies else 0
    
    sweep_results.append({
        'pct_aggressive': pct_agg,
        'avg_steps': avg_steps,
        'std_steps': std_steps,
        'avg_exited': avg_exited,
        'avg_latency': avg_latency
    })
    
    print(f"  {pct_agg*100:4.0f}% aggressive — "
          f"avg steps: {avg_steps:.1f} ± {std_steps:.1f} "
          f"exited: {avg_exited:.1f}")

print("="*50)
print("Aggression sweep complete!")

# Find optimal
optimal = min(sweep_results, key=lambda x: x['avg_steps'])
print(f"\nOPTIMAL AGGRESSION LEVEL: "
      f"{optimal['pct_aggressive']*100:.0f}%")
print(f"Average steps to clear: {optimal['avg_steps']:.1f}")
print(f"Average latency: {optimal['avg_latency']:.1f}")

EXPERIMENT 1 - AGGRESSION SWEEP
Testing aggression levels 1% to 50%
30 runs per level...
     1% aggressive — avg steps: 408.6 ± 82.8 exited: 100.0
     2% aggressive — avg steps: 428.5 ± 94.1 exited: 100.0
     3% aggressive — avg steps: 410.9 ± 70.5 exited: 100.0
     4% aggressive — avg steps: 390.1 ± 77.1 exited: 100.0
     5% aggressive — avg steps: 389.4 ± 64.1 exited: 100.0
     6% aggressive — avg steps: 401.1 ± 61.0 exited: 100.0
     7% aggressive — avg steps: 393.9 ± 58.0 exited: 100.0
     8% aggressive — avg steps: 380.4 ± 74.7 exited: 100.0
     9% aggressive — avg steps: 404.4 ± 73.8 exited: 100.0
    10% aggressive — avg steps: 404.9 ± 76.8 exited: 100.0
    11% aggressive — avg steps: 403.0 ± 92.7 exited: 100.0
    12% aggressive — avg steps: 406.9 ± 78.5 exited: 100.0
    13% aggressive — avg steps: 388.4 ± 76.2 exited: 100.0
    14% aggressive — avg steps: 386.2 ± 48.3 exited: 100.0
    15% aggressive — avg steps: 383.0 ± 59.8 exited: 100.0
    16% aggressive — avg s

In [4]:
# EXPERIMENT 2 - Multi-Scale Aggression Sweep
# Does optimal threshold change with scale?
# Does thrashing appear at higher densities?

print("EXPERIMENT 2 - MULTI-SCALE AGGRESSION SWEEP")
print("="*60)

scale_configs = [
    {
        'name': 'Small',
        'n_agents': 100,
        'width': 20,
        'height': 30,
        'max_steps': 500
    },
    {
        'name': 'Medium',
        'n_agents': 500,
        'width': 45,
        'height': 65,
        'max_steps': 800
    },
    {
        'name': 'Large',
        'n_agents': 1000,
        'width': 65,
        'height': 90,
        'max_steps': 1200
    }
]

# Test aggression levels 5% to 50% in 5% increments
# Keeps runtime manageable while covering full range
aggression_test = [i/100 for i in range(5, 55, 5)]
n_runs = 20  # 20 runs per level per scale

multi_scale_results = {}

for scale in scale_configs:
    print(f"\nScale: {scale['name']} "
          f"({scale['n_agents']} agents)")
    print("-"*60)
    
    scale_data = []
    
    for pct_agg in aggression_test:
        run_steps = []
        run_latencies = []
        
        for run in range(n_runs):
            model = BunchQueueModel(
                n_agents=scale['n_agents'],
                width=scale['width'],
                height=scale['height'],
                pct_aggressive=pct_agg,
                pct_normal=0.25
            )
            exited, _ = model.run(
                max_steps=scale['max_steps']
            )
            run_steps.append(model.steps)
            if model.all_latencies:
                run_latencies.extend(model.all_latencies)
        
        avg_steps = np.mean(run_steps)
        std_steps = np.std(run_steps)
        avg_latency = (np.mean(run_latencies) 
                      if run_latencies else 0)
        
        scale_data.append({
            'pct_aggressive': pct_agg,
            'avg_steps': avg_steps,
            'std_steps': std_steps,
            'avg_latency': avg_latency
        })
        
        print(f"  {pct_agg*100:4.0f}% — "
              f"steps: {avg_steps:.1f} ± {std_steps:.1f} "
              f"latency: {avg_latency:.1f}")
    
    multi_scale_results[scale['name']] = scale_data
    
    # Find optimal for this scale
    optimal = min(scale_data, 
                 key=lambda x: x['avg_steps'])
    print(f"\n  OPTIMAL for {scale['name']}: "
          f"{optimal['pct_aggressive']*100:.0f}% — "
          f"{optimal['avg_steps']:.1f} steps")

print("\n" + "="*60)
print("MULTI-SCALE SWEEP COMPLETE")
print("="*60)

# Summary table
print(f"\n{'Scale':<10} {'Optimal %':<12} {'Best Steps'}")
print("-"*35)
for scale_name, data in multi_scale_results.items():
    optimal = min(data, key=lambda x: x['avg_steps'])
    print(f"{scale_name:<10} "
          f"{optimal['pct_aggressive']*100:.0f}%"
          f"{'':8} "
          f"{optimal['avg_steps']:.1f}")
print("="*60)

EXPERIMENT 2 - MULTI-SCALE AGGRESSION SWEEP

Scale: Small (100 agents)
------------------------------------------------------------
     5% — steps: 398.9 ± 58.7 latency: 124.4
    10% — steps: 407.0 ± 82.9 latency: 114.8
    15% — steps: 395.5 ± 61.5 latency: 108.9
    20% — steps: 406.9 ± 90.1 latency: 107.1
    25% — steps: 391.4 ± 81.4 latency: 99.0
    30% — steps: 349.5 ± 48.5 latency: 90.3
    35% — steps: 350.4 ± 55.2 latency: 88.6
    40% — steps: 326.6 ± 53.3 latency: 80.1
    45% — steps: 366.8 ± 91.8 latency: 78.2
    50% — steps: 332.7 ± 72.6 latency: 71.2

  OPTIMAL for Small: 40% — 326.6 steps

Scale: Medium (500 agents)
------------------------------------------------------------
     5% — steps: 995.6 ± 108.7 latency: 287.4
    10% — steps: 955.1 ± 89.3 latency: 266.3
    15% — steps: 958.6 ± 100.5 latency: 258.0
    20% — steps: 941.7 ± 103.4 latency: 241.2
    25% — steps: 939.5 ± 130.2 latency: 227.2
    30% — steps: 968.1 ± 82.8 latency: 214.7
    35% — steps: 904.